# Objective Quest DATAQUEST 5.0 — V6 Recovery + All-Categorical Sidecar

**Goal:** recover the strongest reproducible V2-style ordinal ensemble first, then test the only V4 representation that was consistently positive: **full raw categorical CatBoost**.

This notebook is intentionally conservative and competition-auditable:

- uses only competition-provided train/test/data-dictionary/sample files;
- no external source reconstruction, no pretrained external model, no manual mapping, no leaderboard probing;
- grouped validation by respondent `id` so duplicated respondents never leak across train/validation;
- no scaling for tree models, no SMOTE/oversampling, no generic imputation;
- compact feature engineering only, with every created feature derived from supplied columns;
- design choices (`feature budget`, `all-cat cardinality cutoff`) are selected only from **train CV probes**;
- final blend weights and QWK thresholds are **cross-fitted**;
- report-only targets `0.48231` and `0.61541` never affect feature/model selection.

## 0. Setup

In [1]:
import os, gc, json, math, random, warnings
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import cohen_kappa_score, roc_auc_score, mean_squared_error, confusion_matrix
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

@dataclass
class Config:
    VERSION: str = 'v6_recovery_allcat'
    TARGET: str = 'life_satisfaction'
    ID_COL: str = 'id'
    HOUSEHOLD_COL: str = 'household_id'
    TRAIN_FILE: str = 'train_final.csv'
    TEST_FILE: str = 'test_final.csv'
    DICT_FILE: str = 'data_dictionary.csv'
    SAMPLE_FILE: str = 'sample_submission.csv'

    N_FOLDS: int = 5
    REPEAT_SEEDS: tuple = (42, 2026)
    PROBE_FOLDS: tuple = (0, 2, 3)
    INNER_FOLDS: int = 4
    USE_GPU: bool = True
    CAT_DEVICES: str = '0:1'

    # Small, evidence-based design probes only.
    FEATURE_BUDGET_CANDIDATES: tuple = (140, 160, 175)
    ALLCAT_CUTOFF_CANDIDATES: tuple = (50, 120)
    PROBE_LGB_ESTIMATORS: int = 1500
    PROBE_CAT_ITERATIONS: int = 1500
    PROBE_EARLY_STOPPING: int = 100

    # Full repeated-CV model budgets.
    FS_ESTIMATORS: int = 1600
    FS_EARLY_STOPPING: int = 100
    LGB_N_ESTIMATORS: int = 5200
    XGB_N_ESTIMATORS: int = 5200
    CAT_ITERATIONS: int = 5200
    LEARNING_RATE: float = 0.025
    EARLY_STOPPING: int = 220
    VERBOSE: int = 500

    # Selected-Cat/XGB categorical cutoff is deliberately conservative.
    SELECTED_CAT_CUTOFF: int = 50

    # Cross-fitted blend selection.
    CORE_WEIGHT_STEP: float = 0.10
    SIDECAR_ALPHA_GRID: tuple = (0.05,0.10,0.15,0.20,0.25,0.30,0.35,0.40,0.45)
    MIN_SIDECAR_GAIN: float = 0.00010
    MAX_SIDECARS: int = 3
    MIN_GLOBAL_SIDECAR_FOLD_SELECTIONS: int = 2

    EXPECTED_TRAIN_ROWS: int = 95609
    EXPECTED_TEST_ROWS: int = 23902
    EXPECTED_UNIQUE_TEST_IDS: int = 23666

    # Report-only references. Never used for tuning.
    OWN_V2_PUBLIC_REFERENCE: float = 0.48193
    NEAREST_PUBLIC_REFERENCE: float = 0.48231
    TOP1_PUBLIC_REFERENCE: float = 0.61541

CFG=Config()
CFG.OUTPUT_DIR = Path('/kaggle/working/oqdq5_v6_recovery_allcat') if Path('/kaggle/working').exists() else Path('./oqdq5_v6_recovery_allcat')
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42); np.random.seed(42)
print(asdict(CFG))

{'VERSION': 'v6_recovery_allcat', 'TARGET': 'life_satisfaction', 'ID_COL': 'id', 'HOUSEHOLD_COL': 'household_id', 'TRAIN_FILE': 'train_final.csv', 'TEST_FILE': 'test_final.csv', 'DICT_FILE': 'data_dictionary.csv', 'SAMPLE_FILE': 'sample_submission.csv', 'N_FOLDS': 5, 'REPEAT_SEEDS': (42, 2026), 'PROBE_FOLDS': (0, 2, 3), 'INNER_FOLDS': 4, 'USE_GPU': True, 'CAT_DEVICES': '0:1', 'FEATURE_BUDGET_CANDIDATES': (140, 160, 175), 'ALLCAT_CUTOFF_CANDIDATES': (50, 120), 'PROBE_LGB_ESTIMATORS': 1500, 'PROBE_CAT_ITERATIONS': 1500, 'PROBE_EARLY_STOPPING': 100, 'FS_ESTIMATORS': 1600, 'FS_EARLY_STOPPING': 100, 'LGB_N_ESTIMATORS': 5200, 'XGB_N_ESTIMATORS': 5200, 'CAT_ITERATIONS': 5200, 'LEARNING_RATE': 0.025, 'EARLY_STOPPING': 220, 'VERBOSE': 500, 'SELECTED_CAT_CUTOFF': 50, 'CORE_WEIGHT_STEP': 0.1, 'SIDECAR_ALPHA_GRID': (0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45), 'MIN_SIDECAR_GAIN': 0.0001, 'MAX_SIDECARS': 3, 'MIN_GLOBAL_SIDECAR_FOLD_SELECTIONS': 2, 'EXPECTED_TRAIN_ROWS': 95609, 'EXPECTED_TEST_

## 1. Problem Understanding & Data Integrity

In [2]:
def locate_data_dir():
    roots=[Path('/kaggle/input'),Path('/mnt/data'),Path('.')]
    required=[CFG.TRAIN_FILE,CFG.TEST_FILE,CFG.SAMPLE_FILE]
    for root in roots:
        if not root.exists():
            continue
        candidates=[root]
        if root.name=='input':
            candidates += [p for p in root.iterdir() if p.is_dir()]
        for d in candidates:
            if d.is_dir() and all((d/f).exists() for f in required):
                return d
        for p in root.rglob(CFG.TRAIN_FILE):
            d=p.parent
            if all((d/f).exists() for f in required):
                return d
    raise FileNotFoundError('Competition CSV files were not found.')

DATA_DIR=locate_data_dir()
train=pd.read_csv(DATA_DIR/CFG.TRAIN_FILE,low_memory=False)
test=pd.read_csv(DATA_DIR/CFG.TEST_FILE,low_memory=False)
sample=pd.read_csv(DATA_DIR/CFG.SAMPLE_FILE,low_memory=False)
data_dictionary=(pd.read_csv(DATA_DIR/CFG.DICT_FILE,low_memory=False)
                 if (DATA_DIR/CFG.DICT_FILE).exists() else pd.DataFrame())

assert len(train)==CFG.EXPECTED_TRAIN_ROWS, train.shape
assert len(test)==CFG.EXPECTED_TEST_ROWS, test.shape
assert CFG.TARGET in train and CFG.TARGET not in test
assert list(test.columns)==[c for c in train.columns if c!=CFG.TARGET]
assert set(train[CFG.TARGET].dropna().astype(int).unique())=={1,2,3,4}

y=train[CFG.TARGET].astype(int).to_numpy()
groups=train[CFG.ID_COL].astype(str).to_numpy()

print('DATA_DIR:',DATA_DIR)
print('train:',train.shape,'test:',test.shape,'sample:',sample.shape)
print('train unique id:',train[CFG.ID_COL].nunique(),'test unique id:',test[CFG.ID_COL].nunique())
print(train[CFG.TARGET].value_counts(normalize=True).sort_index().rename('share'))

DATA_DIR: /kaggle/input/competitions/oq-dq-5
train: (95609, 179) test: (23902, 178) sample: (23902, 2)
train unique id: 94663 test unique id: 23666
life_satisfaction
1    0.449414
2    0.504022
3    0.037225
4    0.009340
Name: share, dtype: float64


In [3]:
def duplicate_id_audit(df,has_target=False):
    vc=df[CFG.ID_COL].value_counts()
    dup_ids=vc[vc>1].index
    rows=df[df[CFG.ID_COL].isin(dup_ids)]
    out={'rows':int(len(df)),'unique_ids':int(df[CFG.ID_COL].nunique()),'duplicate_ids':int(len(dup_ids))}
    if len(rows):
        feature_cols=[c for c in test.columns if c!=CFG.ID_COL]
        payload_max=rows.groupby(CFG.ID_COL)[feature_cols].nunique(dropna=False).max(axis=1)
        out['duplicate_payload_identical']=bool((payload_max<=1).all())
        if has_target:
            out['duplicate_target_consistent']=bool((rows.groupby(CFG.ID_COL)[CFG.TARGET].nunique()<=1).all())
    return out

integrity={
    'train':duplicate_id_audit(train,True),
    'test':duplicate_id_audit(test,False),
    'sample_rows':int(len(sample)),
    'test_unique_ids':int(test[CFG.ID_COL].nunique()),
}
print(json.dumps(integrity,indent=2))
assert integrity['train'].get('duplicate_target_consistent',False)
assert integrity['test'].get('duplicate_payload_identical',False)
assert test[CFG.ID_COL].nunique()==CFG.EXPECTED_UNIQUE_TEST_IDS

{
  "train": {
    "rows": 95609,
    "unique_ids": 94663,
    "duplicate_ids": 946,
    "duplicate_payload_identical": true,
    "duplicate_target_consistent": true
  },
  "test": {
    "rows": 23902,
    "unique_ids": 23666,
    "duplicate_ids": 236,
    "duplicate_payload_identical": true
  },
  "sample_rows": 23902,
  "test_unique_ids": 23666
}


### Validation contract

Every final validation prediction is out-of-fold by respondent `id`.  Two 5-fold grouped repeats reduce variance.  The first repeat supplies the stable fold IDs used for **cross-fitted blend/threshold evaluation**.

In [4]:
survey_year=(train['survey_year'].fillna(-1).astype(str) if 'survey_year' in train else pd.Series('NA',index=train.index))
strata=train[CFG.TARGET].astype(str)+'__yr_'+survey_year

repeated_folds=[]
primary_folds=None
fold_audit=[]
for r,seed in enumerate(CFG.REPEAT_SEEDS):
    cv=StratifiedGroupKFold(n_splits=CFG.N_FOLDS,shuffle=True,random_state=seed)
    folds=list(cv.split(train,strata,groups=groups))
    if r==0: primary_folds=folds
    for f,(tr_idx,va_idx) in enumerate(folds):
        assert not set(groups[tr_idx]).intersection(set(groups[va_idx]))
        repeated_folds.append((r,f,seed,tr_idx,va_idx))
        fold_audit.append({'repeat':r,'fold':f,'train_n':len(tr_idx),'valid_n':len(va_idx),
                           'valid_target_mean':float(y[va_idx].mean())})

display(pd.DataFrame(fold_audit))

,repeat,fold,train_n,valid_n,valid_target_mean
0,0,0,76485,19124,1.606672
1,0,1,76500,19109,1.605893
2,0,2,76487,19122,1.611024
3,0,3,76491,19118,1.604090
4,0,4,76473,19136,1.604776
5,1,0,76501,19108,1.604459
6,1,1,76477,19132,1.605373
7,1,2,76476,19133,1.610255
8,1,3,76483,19126,1.611994
9,1,4,76499,19110,1.600366


## 2. Preprocessing — compact, train-derived, tree-native

In [5]:
# Only supplied column names and train distributions are used.
CONTINUOUS_CANDIDATES=[
    'height_inches','weight_pounds','age','hours_worked_per_week','days_missed_work','income_to_poverty_ratio'
]

def infer_tail_sentinels(series,min_gap=8.0,min_gap_ratio=8.0,max_tail_values=8):
    s=pd.to_numeric(series,errors='coerce').dropna()
    vals=np.sort(s.unique())
    if len(vals)<4: return []
    gaps=np.diff(vals); positive=gaps[gaps>0]
    if not len(positive): return []
    typical=float(np.median(positive)); i=int(np.argmax(gaps)); gap=float(gaps[i]); tail=vals[i+1:]
    if gap<min_gap: return []
    if typical>0 and gap/typical<min_gap_ratio: return []
    if len(tail)>max_tail_values: return []
    return [float(v) for v in tail]

SENTINEL_MAP={}
sentinel_rows=[]
for c in CONTINUOUS_CANDIDATES:
    if c in train:
        vals=infer_tail_sentinels(train[c])
        SENTINEL_MAP[c]=vals
        sentinel_rows.append({'feature':c,'tail_values':vals,
                              'train_support':int(pd.to_numeric(train[c],errors='coerce').isin(vals).sum()) if vals else 0,
                              'test_support':int(pd.to_numeric(test[c],errors='coerce').isin(vals).sum()) if vals else 0})
sentinel_audit=pd.DataFrame(sentinel_rows)
display(sentinel_audit)

,feature,tail_values,train_support,test_support
0,height_inches,"[96.0, 97.0, 98.0, 99.0]",6166,1520
1,weight_pounds,"[996.0, 997.0, 999.0]",7484,1834
2,age,"[97.0, 99.0]",186,47
3,hours_worked_per_week,[990.0],482,133
4,days_missed_work,"[997.0, 998.0, 999.0]",459,106
5,income_to_poverty_ratio,[110.0],485,120


In [6]:
FUNCTIONAL_DIFFICULTY_COLS=[
    'difficulty_walking_steps','difficulty_walking_short','difficulty_walking_medium','difficulty_climbing_steps',
    'difficulty_using_hands','difficulty_raising_arm','difficulty_self_care','difficulty_communicating',
    'difficulty_remembering','difficulty_errands_alone','difficulty_social_activities']
CHRONIC_CONDITION_COLS=[
    'hypertension_ever','high_cholesterol_ever','heart_disease_ever','heart_attack_ever','angina_ever','stroke_ever',
    'asthma_ever','diabetes_ever','prediabetes_ever','cancer_ever','lung_disease_ever','arthritis_ever','dementia_ever','chronic_fatigue_ever']
COST_BARRIER_COLS=[
    'trouble_paying_medical_bills','delayed_care_due_to_cost','skipped_care_due_to_cost',
    'skipped_medication_to_save_money','took_less_medication_to_save_money','delayed_prescription_to_save_money',
    'skipped_prescription_due_to_cost','delayed_therapy_due_to_cost','skipped_therapy_due_to_cost']
INCOME_SOURCE_COLS=['has_wage_income','has_investment_income','has_social_security_income','has_disability_income',
                    'has_public_assistance_income','has_retirement_income','has_other_income']
INSURANCE_COLS=['has_health_insurance','has_private_insurance','has_senior_gov_insurance','has_lowincome_gov_insurance',
                'has_child_gov_insurance','has_military_insurance','has_state_health_plan','has_other_gov_insurance']

STABLE_FORCE_KEEP=[
    'general_health_status','depression_frequency','anxiety_frequency','income_to_poverty_ratio','marital_status',
    'marital_status_detailed','difficulty_social_activities','difficulty_remembering','worry_about_medical_costs',
    'food_security_4cat','age','survey_weight','row_missing_count','special_code_count',
    'functional_difficulty_count','chronic_condition_count','cost_barrier_count','income_source_count','insurance_count',
    'mental_frequency_burden','food_insecurity_burden','household_size_proxy'
]

FEATURE_PROVENANCE=pd.DataFrame([
    ('row_missing_count','count NaN across supplied predictors','unsupervised row state'),
    ('special_code_count','count of train-gap inferred tail flags','train value distribution'),
    ('functional_difficulty_count','count supplied difficulty fields equal to code 1','supplied column family'),
    ('chronic_condition_count','count supplied chronic-condition fields equal to code 1','supplied column family'),
    ('cost_barrier_count','count supplied cost-barrier fields equal to code 1','supplied column family'),
    ('income_source_count','count supplied income-source fields equal to code 1','supplied column family'),
    ('insurance_count','count supplied insurance-source fields equal to code 1','supplied column family'),
    ('mental_frequency_burden','reverse compact 1..5 depression/anxiety frequencies','train-observed compact codes'),
    ('food_insecurity_burden','reverse compact 1..3 food-hardship codes','train-observed compact codes'),
    ('household_size_proxy','children_in_household + adults_in_household','supplied household counts'),
],columns=['feature','formula','evidence'])
display(FEATURE_PROVENANCE)

,feature,formula,evidence
0,row_missing_count,count NaN across supplied predictors,unsupervised row state
1,special_code_count,count of train-gap inferred tail flags,train value distribution
2,functional_difficulty_count,count supplied difficulty fields equal to code 1,supplied column family
3,chronic_condition_count,count supplied chronic-condition fields equal ...,supplied column family
4,cost_barrier_count,count supplied cost-barrier fields equal to co...,supplied column family
5,income_source_count,count supplied income-source fields equal to c...,supplied column family
6,insurance_count,count supplied insurance-source fields equal t...,supplied column family
7,mental_frequency_burden,reverse compact 1..5 depression/anxiety freque...,train-observed compact codes
8,food_insecurity_burden,reverse compact 1..3 food-hardship codes,train-observed compact codes
9,household_size_proxy,children_in_household + adults_in_household,supplied household counts


In [7]:
def count_code_one(df,cols):
    cols=[c for c in cols if c in df]
    return (df[cols]==1).sum(axis=1).astype(np.int16) if cols else pd.Series(0,index=df.index,dtype=np.int16)

def reverse_compact_ordinal(s,max_valid):
    x=pd.to_numeric(s,errors='coerce'); out=pd.Series(np.nan,index=x.index,dtype=float)
    m=x.between(1,max_valid); out.loc[m]=(max_valid+1)-x.loc[m]
    return out

def build_numeric_view(df):
    out=df.drop(columns=[CFG.TARGET],errors='ignore').copy()
    raw=out.drop(columns=[CFG.ID_COL,CFG.HOUSEHOLD_COL],errors='ignore')
    out['row_missing_count']=raw.isna().sum(axis=1).astype(np.int16)

    flags=[]
    for c,vals in SENTINEL_MAP.items():
        if c in out and vals:
            s=pd.to_numeric(out[c],errors='coerce'); flag=s.isin(vals); name=f'{c}__train_tail_flag'
            out[name]=flag.astype(np.int8); flags.append(name); out.loc[flag,c]=np.nan
    out['special_code_count']=out[flags].sum(axis=1).astype(np.int16) if flags else 0

    out['functional_difficulty_count']=count_code_one(out,FUNCTIONAL_DIFFICULTY_COLS)
    out['chronic_condition_count']=count_code_one(out,CHRONIC_CONDITION_COLS)
    out['cost_barrier_count']=count_code_one(out,COST_BARRIER_COLS)
    out['income_source_count']=count_code_one(out,INCOME_SOURCE_COLS)
    out['insurance_count']=count_code_one(out,INSURANCE_COLS)

    food=[]
    for c in ['worried_food_would_run_out','food_did_not_last','could_not_afford_balanced_meals']:
        if c in out: food.append(reverse_compact_ordinal(out[c],3).fillna(0))
    out['food_insecurity_burden']=sum(food).astype(np.float32) if food else 0.0

    mental=[]
    for c in ['depression_frequency','anxiety_frequency']:
        if c in out: mental.append(reverse_compact_ordinal(out[c],5).fillna(0))
    out['mental_frequency_burden']=sum(mental).astype(np.float32) if mental else 0.0

    hh=[]
    for c in ['children_in_household','adults_in_household']:
        if c in out: hh.append(pd.to_numeric(out[c],errors='coerce').fillna(0))
    out['household_size_proxy']=sum(hh).astype(np.float32) if hh else 0.0

    out=out.drop(columns=[CFG.ID_COL,CFG.HOUSEHOLD_COL],errors='ignore')
    for c in out:
        if not pd.api.types.is_numeric_dtype(out[c]): out[c]=pd.to_numeric(out[c],errors='coerce')
    return out

X_num=build_numeric_view(train)
X_test_num=build_numeric_view(test)
assert list(X_num.columns)==list(X_test_num.columns)
print('Numeric compact view:',X_num.shape)

Numeric compact view: (95609, 192)


## 3. Explainable-AI / EDA checks

In [8]:
# Compact EDA: target balance, missingness, and strongest rank correlations.
display(train[CFG.TARGET].value_counts(normalize=True).sort_index().rename('target_share').to_frame())

missing=pd.DataFrame({'train_missing':train.drop(columns=[CFG.TARGET]).isna().mean(),
                      'test_missing':test.isna().mean()})
missing['abs_gap']=(missing.train_missing-missing.test_missing).abs()
display(missing.sort_values('train_missing',ascending=False).head(25))

corr_cols=[c for c in X_num.columns if X_num[c].nunique(dropna=True)>1]
corr_to_target=[]
for c in corr_cols:
    try:
        v=pd.to_numeric(X_num[c],errors='coerce')
        corr_to_target.append((c,float(pd.Series(v).corr(pd.Series(y),method='spearman'))))
    except Exception:
        pass
corr_to_target=pd.DataFrame(corr_to_target,columns=['feature','spearman']).assign(abs_s=lambda d:d.spearman.abs()).sort_values('abs_s',ascending=False)
display(corr_to_target.head(30))

,target_share
life_satisfaction,
1,0.449414
2,0.504022
3,0.037225
4,0.009340


,train_missing,test_missing,abs_gap
flu_vaccine_year,0.538485,0.542800,0.004315
flu_vaccine_month,0.534291,0.540833,0.006543
ever_married,0.521112,0.520919,0.000193
gestational_diabetes_ever,0.500152,0.492051,0.008101
works_full_time,0.485854,0.485692,0.000162
hours_worked_per_week,0.485080,0.484102,0.000978
insurance_marketplace_plan1,0.449884,0.452180,0.002295
days_missed_work,0.440147,0.437871,0.002276
has_paid_sick_leave,0.439645,0.439628,0.000016
employer_offers_insurance,0.439341,0.435361,0.003980


,feature,spearman,abs_s
140,general_health_status,0.361369,0.361369
64,depression_frequency,-0.322500,0.322500
186,mental_frequency_burden,0.282600,0.282600
70,anxiety_frequency,-0.259551,0.259551
5,difficulty_social_activities,0.229138,0.229138
45,depression_ever,-0.227277,0.227277
3,income_to_poverty_ratio,-0.220997,0.220997
127,difficulty_remembering,0.216128,0.216128
180,functional_difficulty_count,-0.215489,0.215489
18,income_poverty_category,-0.213499,0.213499


## 4. Metrics, calibration, and fold-safe feature selection

QWK thresholds are never tuned on the validation fold being scored.  Design probes use an inner calibration split; final model evaluation uses cross-fitted calibration across the five primary folds.

In [9]:
def qwk(y_true,y_pred):
    return cohen_kappa_score(np.asarray(y_true),np.asarray(y_pred),weights='quadratic')

def rmse(y_true,pred):
    return float(mean_squared_error(y_true,pred)**0.5)

def apply_thresholds(pred,thresholds):
    return np.digitize(np.asarray(pred,float),np.sort(np.asarray(thresholds,float))).astype(int)+1

def initial_thresholds(y_true,pred):
    dist=pd.Series(y_true).value_counts(normalize=True).sort_index()
    cum=np.cumsum([dist.get(k,0.0) for k in [1,2,3]])
    return np.quantile(np.asarray(pred,float),np.clip(cum,0.001,0.999))

def optimize_thresholds(y_true,pred,maxiter=160):
    y_true=np.asarray(y_true); pred=np.asarray(pred,float)
    starts=[initial_thresholds(y_true,pred),np.array([1.55,2.03,2.52])]
    best=None
    for start in starts:
        def obj(t):
            t=np.sort(t)
            if np.min(np.diff(t))<0.025: return 1e3
            return -qwk(y_true,apply_thresholds(pred,t))
        res=minimize(obj,np.sort(start),method='Nelder-Mead',options={'maxiter':maxiter,'xatol':1e-5,'fatol':1e-7})
        t=np.sort(res.x); s=qwk(y_true,apply_thresholds(pred,t))
        if best is None or s>best[1]: best=(t,s)
    return best

def boundary_auc(y_true,pred):
    y_true=np.asarray(y_true); pred=np.asarray(pred,float)
    out={}
    for k in (1,2,3): out[f'auc_gt{k}']=float(roc_auc_score((y_true>k).astype(int),pred))
    mask=np.isin(y_true,[1,2])
    out['auc_1_vs_2']=float(roc_auc_score((y_true[mask]==2).astype(int),pred[mask]))
    return out

def inner_fit_cal_split(outer_train_idx,seed):
    idx=np.asarray(outer_train_idx)
    yy=y[idx]; gg=groups[idx]
    sy=(train.iloc[idx]['survey_year'].fillna(-1).astype(str).to_numpy() if 'survey_year' in train else np.array(['NA']*len(idx)))
    ss=pd.Series(yy).astype(str).to_numpy()+'__yr_'+sy
    cv=StratifiedGroupKFold(n_splits=CFG.INNER_FOLDS,shuffle=True,random_state=seed)
    a,b=next(cv.split(idx,ss,groups=gg))
    return idx[a],idx[b]

In [10]:
def make_lgb_fs(seed,use_gpu=True,n_estimators=None):
    p=dict(objective='regression_l2',n_estimators=n_estimators or CFG.FS_ESTIMATORS,learning_rate=0.035,
           num_leaves=31,min_child_samples=42,colsample_bytree=0.88,subsample=0.90,subsample_freq=1,
           reg_lambda=5.0,random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: p.update(device_type='gpu',max_bin=63)
    return lgb.LGBMRegressor(**p)

def fit_lgb_fs(Xtr,ytr,Xv,yv,seed,n_estimators=None):
    last=None
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_lgb_fs(seed,gpu,n_estimators)
            m.fit(Xtr,ytr,eval_set=[(Xv,yv)],callbacks=[lgb.early_stopping(CFG.FS_EARLY_STOPPING,verbose=False)])
            return m
        except Exception as e:
            last=e
            if gpu:
                print('FS LightGBM GPU fallback:',repr(e)); gc.collect(); continue
            raise
    raise last

def select_features_grouped(outer_idx,budget,seed):
    fit_idx,cal_idx=inner_fit_cal_split(outer_idx,seed)
    m=fit_lgb_fs(X_num.iloc[fit_idx],y[fit_idx],X_num.iloc[cal_idx],y[cal_idx],seed)
    gain=pd.Series(m.booster_.feature_importance(importance_type='gain'),index=X_num.columns).sort_values(ascending=False)
    forced=[c for c in STABLE_FORCE_KEEP if c in X_num.columns]
    selected=list(dict.fromkeys(forced+[c for c in gain.index if c not in forced]))[:min(budget,len(X_num.columns))]
    return selected,gain

## 5. Model factories

In [11]:
def make_lgb_reg(seed,use_gpu=True,n_estimators=None):
    p=dict(objective='regression_l2',n_estimators=n_estimators or CFG.LGB_N_ESTIMATORS,
           learning_rate=CFG.LEARNING_RATE,num_leaves=31,min_child_samples=40,max_depth=-1,
           subsample=0.88,subsample_freq=1,colsample_bytree=0.86,reg_alpha=0.04,reg_lambda=4.5,
           random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: p.update(device_type='gpu',max_bin=63)
    return lgb.LGBMRegressor(**p)

def make_lgb_binary(seed,use_gpu=True):
    p=dict(objective='binary',n_estimators=CFG.LGB_N_ESTIMATORS,learning_rate=CFG.LEARNING_RATE,
           num_leaves=31,min_child_samples=40,subsample=0.88,subsample_freq=1,colsample_bytree=0.86,
           reg_lambda=4.5,random_state=seed,n_jobs=-1,verbosity=-1)
    if use_gpu: p.update(device_type='gpu',max_bin=63)
    return lgb.LGBMClassifier(**p)

def make_xgb_reg(seed,use_gpu=True,enable_categorical=False):
    p=dict(objective='reg:squarederror',eval_metric='rmse',n_estimators=CFG.XGB_N_ESTIMATORS,
           learning_rate=CFG.LEARNING_RATE,max_depth=6,min_child_weight=8,subsample=0.86,
           colsample_bytree=0.86,reg_alpha=0.04,reg_lambda=4.5,tree_method='hist',
           random_state=seed,n_jobs=-1,early_stopping_rounds=CFG.EARLY_STOPPING,
           enable_categorical=enable_categorical)
    if enable_categorical:
        p.update(max_cat_to_onehot=4,max_cat_threshold=64)
    if use_gpu: p['device']='cuda:0'
    return xgb.XGBRegressor(**p)

def make_cat(seed,use_gpu=True,iterations=None):
    p=dict(loss_function='RMSE',eval_metric='RMSE',iterations=iterations or CFG.CAT_ITERATIONS,
           learning_rate=CFG.LEARNING_RATE,depth=7,l2_leaf_reg=6.0,random_strength=0.4,
           random_seed=seed,od_type='Iter',od_wait=CFG.EARLY_STOPPING,
           allow_writing_files=False,verbose=CFG.VERBOSE)
    if use_gpu: p.update(task_type='GPU',devices=CFG.CAT_DEVICES)
    else: p.update(task_type='CPU',thread_count=-1)
    return CatBoostRegressor(**p)

In [12]:
def fit_lgb_reg(Xtr,ytr,Xv,yv,Xt,seed,n_estimators=None):
    last=None
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_lgb_reg(seed,gpu,n_estimators)
            m.fit(Xtr,ytr,eval_set=[(Xv,yv)],eval_metric='rmse',
                  callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING,verbose=False),lgb.log_evaluation(CFG.VERBOSE)])
            return m,m.predict(Xv,num_iteration=m.best_iteration_),m.predict(Xt,num_iteration=m.best_iteration_)
        except Exception as e:
            last=e
            if gpu:
                print('LightGBM GPU fallback:',repr(e)); gc.collect(); continue
            raise
    raise last

def fit_xgb_reg(Xtr,ytr,Xv,yv,Xt,seed,enable_categorical=False):
    last=None
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_xgb_reg(seed,gpu,enable_categorical)
            m.fit(Xtr,ytr,eval_set=[(Xv,yv)],verbose=False)
            return m,m.predict(Xv),m.predict(Xt)
        except Exception as e:
            last=e
            if gpu:
                print('XGBoost CUDA fallback:',repr(e)); gc.collect(); continue
            raise
    raise last

def fit_cat_reg(Xtr,ytr,Xv,yv,Xt,cat_cols,seed,iterations=None,probe=False):
    last=None
    for gpu in ([True,False] if CFG.USE_GPU else [False]):
        try:
            m=make_cat(seed,gpu,iterations)
            m.fit(Xtr,ytr,eval_set=(Xv,yv),cat_features=cat_cols,use_best_model=True,
                  verbose=False if probe else CFG.VERBOSE)
            return m,m.predict(Xv),m.predict(Xt)
        except Exception as e:
            last=e
            if gpu:
                print('CatBoost GPU fallback:',repr(e)); gc.collect(); continue
            raise
    raise last

def fit_cumulative_ordinal(Xtr,ytr,Xv,yv,Xt,seed):
    pv=[]; pt=[]; models=[]
    for boundary in (1,2,3):
        yt=(np.asarray(ytr)>boundary).astype(int); yvv=(np.asarray(yv)>boundary).astype(int)
        fitted=False
        for gpu in ([True,False] if CFG.USE_GPU else [False]):
            try:
                m=make_lgb_binary(seed+boundary*101,gpu)
                m.fit(Xtr,yt,eval_set=[(Xv,yvv)],eval_metric='binary_logloss',
                      callbacks=[lgb.early_stopping(CFG.EARLY_STOPPING,verbose=False)])
                pv.append(m.predict_proba(Xv,num_iteration=m.best_iteration_)[:,1])
                pt.append(m.predict_proba(Xt,num_iteration=m.best_iteration_)[:,1])
                models.append(m); fitted=True; break
            except Exception as e:
                if gpu:
                    print(f'Cumulative boundary {boundary} GPU fallback:',repr(e)); gc.collect(); continue
                raise
        assert fitted
    pv=np.column_stack(pv); pt=np.column_stack(pt)
    # Enforce P(y>1) >= P(y>2) >= P(y>3).
    pv[:,1]=np.minimum(pv[:,1],pv[:,0]); pv[:,2]=np.minimum(pv[:,2],pv[:,1])
    pt[:,1]=np.minimum(pt[:,1],pt[:,0]); pt[:,2]=np.minimum(pt[:,2],pt[:,1])
    return models,1.0+pv.sum(axis=1),1.0+pt.sum(axis=1)

## 6. Representation helpers

In [13]:
RAW_FEATURES=[c for c in test.columns if c not in [CFG.ID_COL,CFG.HOUSEHOLD_COL]]

def prepare_selected_cat(train_idx,df_idx,selected,cutoff=50,other_df=None):
    # selected numeric/engineered frame; only original low-card supplied columns become categorical.
    ref=X_num.iloc[train_idx]
    src=X_test_num if other_df is not None else X_num
    out=src.iloc[df_idx][selected].copy() if other_df is None else X_test_num[selected].copy()
    cat=[]
    for c in selected:
        if c in RAW_FEATURES and ref[c].nunique(dropna=True)<=cutoff:
            cat.append(c); out[c]=out[c].map(lambda v:'__MISSING__' if pd.isna(v) else str(v))
    return out,cat

def prepare_xgb_cat_views(train_idx,val_idx,selected):
    A=X_num.iloc[train_idx][selected].copy(); B=X_num.iloc[val_idx][selected].copy(); C=X_test_num[selected].copy()
    cat=[]
    for c in selected:
        if c not in RAW_FEATURES: continue
        if A[c].nunique(dropna=True)>CFG.SELECTED_CAT_CUTOFF: continue
        levels=pd.Index(A[c].dropna().astype(str).unique()).tolist()
        for frame in (A,B,C):
            vals=frame[c].map(lambda v:'__MISSING__' if pd.isna(v) else str(v))
            cats=levels+(['__MISSING__'] if '__MISSING__' not in levels else [])
            frame[c]=pd.Categorical(vals,categories=cats)
        cat.append(c)
    return A,B,C,cat

def derive_allcat_schema(fit_idx,cutoff):
    ref=train.iloc[fit_idx]
    cat=[c for c in RAW_FEATURES if ref[c].nunique(dropna=True)<=cutoff]
    cont=[c for c in RAW_FEATURES if c not in cat]
    return cat,cont

def transform_allcat(df,cat_cols,cont_cols):
    out=df[RAW_FEATURES].copy()
    for c in cat_cols:
        out[c]=out[c].map(lambda v:'__MISSING__' if pd.isna(v) else str(v))
    for c in cont_cols:
        out[c]=pd.to_numeric(out[c],errors='coerce')
        vals=SENTINEL_MAP.get(c,[])
        if vals: out.loc[out[c].isin(vals),c]=np.nan
    out['row_missing_count']=df[RAW_FEATURES].isna().sum(axis=1).astype(np.float32)
    return out

## 7. Train-only design probes

These probes choose only two design knobs that already had evidence in earlier train-CV work:

1. compact selected-feature budget: `140 / 160 / 175`;
2. all-categorical CatBoost cardinality cutoff: `50 / 120`.

Each probe uses an **outer untouched fold** plus a separate inner fit/calibration split.  No leaderboard score participates.

In [14]:
probe_rows=[]
for f in CFG.PROBE_FOLDS:
    outer_tr,outer_va=primary_folds[f]
    fit_idx,cal_idx=inner_fit_cal_split(outer_tr,7000+f)

    # One ranking, then evaluate the three nested budgets on the same honest fit/cal/outer-valid structure.
    fs_model=fit_lgb_fs(X_num.iloc[fit_idx],y[fit_idx],X_num.iloc[cal_idx],y[cal_idx],7100+f,CFG.PROBE_LGB_ESTIMATORS)
    gain=pd.Series(fs_model.booster_.feature_importance(importance_type='gain'),index=X_num.columns).sort_values(ascending=False)
    forced=[c for c in STABLE_FORCE_KEEP if c in X_num.columns]
    ranking=list(dict.fromkeys(forced+[c for c in gain.index if c not in forced]))

    for budget in CFG.FEATURE_BUDGET_CANDIDATES:
        selected=ranking[:min(budget,len(ranking))]
        m,pcal,_=fit_lgb_reg(X_num.iloc[fit_idx][selected],y[fit_idx],X_num.iloc[cal_idx][selected],y[cal_idx],
                             X_num.iloc[outer_va][selected],7200+f+budget,n_estimators=CFG.PROBE_LGB_ESTIMATORS)
        pva=m.predict(X_num.iloc[outer_va][selected],num_iteration=m.best_iteration_)
        thr,_=optimize_thresholds(y[cal_idx],pcal,maxiter=120)
        qq=qwk(y[outer_va],apply_thresholds(pva,thr))
        probe_rows.append({'probe':'feature_budget','value':int(budget),'fold':int(f),'outer_qwk':float(qq)})
        del m; gc.collect()

    for cutoff in CFG.ALLCAT_CUTOFF_CANDIDATES:
        cat_cols,cont_cols=derive_allcat_schema(fit_idx,cutoff)
        Af=transform_allcat(train.iloc[fit_idx],cat_cols,cont_cols)
        Ac=transform_allcat(train.iloc[cal_idx],cat_cols,cont_cols)
        Av=transform_allcat(train.iloc[outer_va],cat_cols,cont_cols)
        m,pcal,_=fit_cat_reg(Af,y[fit_idx],Ac,y[cal_idx],Av,cat_cols,7300+f+cutoff,
                             iterations=CFG.PROBE_CAT_ITERATIONS,probe=True)
        pva=m.predict(Av)
        thr,_=optimize_thresholds(y[cal_idx],pcal,maxiter=120)
        qq=qwk(y[outer_va],apply_thresholds(pva,thr))
        probe_rows.append({'probe':'allcat_cutoff','value':int(cutoff),'fold':int(f),'outer_qwk':float(qq),
                           'n_categorical':int(len(cat_cols))})
        del m,Af,Ac,Av; gc.collect()

probe_df=pd.DataFrame(probe_rows)
probe_summary=probe_df.groupby(['probe','value'],as_index=False).agg(mean_qwk=('outer_qwk','mean'),wins=('outer_qwk','count'),std_qwk=('outer_qwk','std'))
display(probe_summary.sort_values(['probe','mean_qwk'],ascending=[True,False]))

BUDGET=int(probe_summary[probe_summary.probe=='feature_budget'].sort_values(['mean_qwk','value'],ascending=[False,True]).iloc[0].value)
ALLCAT_CUTOFF=int(probe_summary[probe_summary.probe=='allcat_cutoff'].sort_values(['mean_qwk','value'],ascending=[False,True]).iloc[0].value)
print('SELECTED FEATURE BUDGET =',BUDGET)
print('SELECTED ALL-CAT CUTOFF =',ALLCAT_CUTOFF)

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[500]	valid_0's rmse: 0.520091	valid_0's l2: 0.270495
[500]	valid_0's rmse: 0.519739	valid_0's l2: 0.270129
[500]	valid_0's rmse: 0.519948	valid_0's l2: 0.270346
[500]	valid_0's rmse: 0.517728	valid_0's l2: 0.268043
[500]	valid_0's rmse: 0.517799	valid_0's l2: 0.268116
[500]	valid_0's rmse: 0.517803	valid_0's l2: 0.26812
[500]	valid_0's rmse: 0.514123	valid_0's l2: 0.264322
[500]	valid_0's rmse: 0.513945	valid_0's l2: 0.26414
[500]	valid_0's rmse: 0.513671	valid_0's l2: 0.263858


,probe,value,mean_qwk,wins,std_qwk
0,allcat_cutoff,50,0.475560,3,0.014108
1,allcat_cutoff,120,0.474425,3,0.011934
2,feature_budget,140,0.474689,3,0.012354
3,feature_budget,160,0.474658,3,0.013129
4,feature_budget,175,0.474338,3,0.013596


SELECTED FEATURE BUDGET = 140
SELECTED ALL-CAT CUTOFF = 50


## 8. Full 2×5 repeated grouped CV

Core:

- LightGBM regression
- XGBoost regression
- selected-feature CatBoost regression

Sidecars:

- XGBoost native categorical representation (V2 objective/representation diversity)
- LightGBM cumulative ordinal expected score (V2's strongest objective-diversity idea)
- **full raw all-categorical CatBoost** (V4's only 3/3 positive representation candidate)

In [15]:
branch_names=['lgb_core','xgb_core','cat_core','xgb_categorical','lgb_cumulative_ordinal','catboost_allcat']
OOF_SUM={b:np.zeros(len(train),dtype=np.float64) for b in branch_names}
OOF_COUNT={b:np.zeros(len(train),dtype=np.int8) for b in branch_names}
TEST_SUM={b:np.zeros(len(test),dtype=np.float64) for b in branch_names}
branch_failures=defaultdict(list)
feature_selection_rows=[]
feature_importance_rows=[]
primary_fold_id=np.full(len(train),-1,dtype=np.int8)

for repeat,fold,seed,tr_idx,va_idx in repeated_folds:
    print('\n'+'='*96)
    print(f'REPEAT {repeat+1}/{len(CFG.REPEAT_SEEDS)} | FOLD {fold+1}/{CFG.N_FOLDS}')
    print('='*96)
    if repeat==0: primary_fold_id[va_idx]=fold

    selected,fs_gain=select_features_grouped(tr_idx,BUDGET,seed+1000+fold*31)
    feature_selection_rows.append({'repeat':repeat,'fold':fold,'budget':BUDGET,'selected_count':len(selected),
                                   'selected_features':'|'.join(selected)})
    A=X_num.iloc[tr_idx][selected]; B=X_num.iloc[va_idx][selected]; C=X_test_num[selected]
    model_seed=seed+fold*997

    # Core LGB
    m,pv,pt=fit_lgb_reg(A,y[tr_idx],B,y[va_idx],C,model_seed)
    OOF_SUM['lgb_core'][va_idx]+=pv; OOF_COUNT['lgb_core'][va_idx]+=1; TEST_SUM['lgb_core']+=pt
    for feat,imp in zip(selected,m.booster_.feature_importance(importance_type='gain')):
        feature_importance_rows.append({'repeat':repeat,'fold':fold,'branch':'lgb_core','feature':feat,'importance':float(imp)})
    del m; gc.collect()

    # Core XGB numeric
    m,pv,pt=fit_xgb_reg(A,y[tr_idx],B,y[va_idx],C,model_seed+11,False)
    OOF_SUM['xgb_core'][va_idx]+=pv; OOF_COUNT['xgb_core'][va_idx]+=1; TEST_SUM['xgb_core']+=pt
    for feat,imp in zip(selected,m.feature_importances_):
        feature_importance_rows.append({'repeat':repeat,'fold':fold,'branch':'xgb_core','feature':feat,'importance':float(imp)})
    del m; gc.collect()

    # Core CatBoost on selected view; original low-card selected columns are explicit categoricals.
    Act,cat_sel=prepare_selected_cat(tr_idx,tr_idx,selected,CFG.SELECTED_CAT_CUTOFF)
    Bct,_=prepare_selected_cat(tr_idx,va_idx,selected,CFG.SELECTED_CAT_CUTOFF)
    Cct=X_test_num[selected].copy()
    for c in cat_sel: Cct[c]=Cct[c].map(lambda v:'__MISSING__' if pd.isna(v) else str(v))
    m,pv,pt=fit_cat_reg(Act,y[tr_idx],Bct,y[va_idx],Cct,cat_sel,model_seed+23)
    OOF_SUM['cat_core'][va_idx]+=pv; OOF_COUNT['cat_core'][va_idx]+=1; TEST_SUM['cat_core']+=pt
    try:
        for feat,imp in zip(Act.columns,m.get_feature_importance()):
            feature_importance_rows.append({'repeat':repeat,'fold':fold,'branch':'cat_core','feature':feat,'importance':float(imp)})
    except Exception: pass
    del m,Act,Bct,Cct; gc.collect()

    # V2 categorical XGBoost sidecar.
    try:
        Ac,Bc,Cc,xgb_cat_cols=prepare_xgb_cat_views(tr_idx,va_idx,selected)
        m,pv,pt=fit_xgb_reg(Ac,y[tr_idx],Bc,y[va_idx],Cc,model_seed+41,True)
        OOF_SUM['xgb_categorical'][va_idx]+=pv; OOF_COUNT['xgb_categorical'][va_idx]+=1; TEST_SUM['xgb_categorical']+=pt
        del m,Ac,Bc,Cc; gc.collect()
    except Exception as e:
        branch_failures['xgb_categorical'].append((repeat,fold,repr(e))); print('SKIP xgb_categorical:',repr(e)); gc.collect()

    # V2 cumulative ordinal sidecar.
    try:
        models,pv,pt=fit_cumulative_ordinal(A,y[tr_idx],B,y[va_idx],C,model_seed+79)
        OOF_SUM['lgb_cumulative_ordinal'][va_idx]+=pv; OOF_COUNT['lgb_cumulative_ordinal'][va_idx]+=1; TEST_SUM['lgb_cumulative_ordinal']+=pt
        del models; gc.collect()
    except Exception as e:
        branch_failures['lgb_cumulative_ordinal'].append((repeat,fold,repr(e))); print('SKIP ordinal:',repr(e)); gc.collect()

    # V4 full raw categorical CatBoost sidecar.
    try:
        allcat_cols,allcont_cols=derive_allcat_schema(tr_idx,ALLCAT_CUTOFF)
        Ar=transform_allcat(train.iloc[tr_idx],allcat_cols,allcont_cols)
        Br=transform_allcat(train.iloc[va_idx],allcat_cols,allcont_cols)
        Cr=transform_allcat(test,allcat_cols,allcont_cols)
        m,pv,pt=fit_cat_reg(Ar,y[tr_idx],Br,y[va_idx],Cr,allcat_cols,model_seed+131)
        OOF_SUM['catboost_allcat'][va_idx]+=pv; OOF_COUNT['catboost_allcat'][va_idx]+=1; TEST_SUM['catboost_allcat']+=pt
        del m,Ar,Br,Cr; gc.collect()
    except Exception as e:
        branch_failures['catboost_allcat'].append((repeat,fold,repr(e))); print('SKIP catboost_allcat:',repr(e)); gc.collect()

assert (primary_fold_id>=0).all()
print('Branch failures:')
for b,v in branch_failures.items(): print(b,len(v),v[:2])


REPEAT 1/2 | FOLD 1/5
[500]	valid_0's rmse: 0.513188	valid_0's l2: 0.263362
0:	learn: 0.6041554	test: 0.6055015	best: 0.6055015 (0)	total: 114ms	remaining: 9m 53s
500:	learn: 0.5095947	test: 0.5144863	best: 0.5144837 (499)	total: 32.7s	remaining: 5m 6s
1000:	learn: 0.5031227	test: 0.5133635	best: 0.5133631 (999)	total: 1m 4s	remaining: 4m 31s
1500:	learn: 0.4974815	test: 0.5129473	best: 0.5129473 (1500)	total: 1m 37s	remaining: 3m 59s
2000:	learn: 0.4923951	test: 0.5128183	best: 0.5128090 (1961)	total: 2m 9s	remaining: 3m 27s
2500:	learn: 0.4872677	test: 0.5127202	best: 0.5127182 (2496)	total: 2m 42s	remaining: 2m 55s
3000:	learn: 0.4821934	test: 0.5126184	best: 0.5126017 (2980)	total: 3m 16s	remaining: 2m 24s
3500:	learn: 0.4769629	test: 0.5125666	best: 0.5125323 (3404)	total: 3m 51s	remaining: 1m 52s
bestTest = 0.512532255
bestIteration = 3404
Shrink model to first 3405 iterations.


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


0:	learn: 0.6039805	test: 0.6052767	best: 0.6052767 (0)	total: 96.1ms	remaining: 8m 19s
500:	learn: 0.5105168	test: 0.5145309	best: 0.5145309 (500)	total: 36.9s	remaining: 5m 45s
1000:	learn: 0.5046625	test: 0.5137392	best: 0.5137392 (1000)	total: 1m 15s	remaining: 5m 14s
1500:	learn: 0.4996890	test: 0.5133436	best: 0.5133417 (1499)	total: 1m 54s	remaining: 4m 41s
2000:	learn: 0.4949485	test: 0.5131948	best: 0.5131888 (1971)	total: 2m 34s	remaining: 4m 7s
2500:	learn: 0.4898919	test: 0.5130361	best: 0.5130302 (2476)	total: 3m 15s	remaining: 3m 31s
bestTest = 0.5130241526
bestIteration = 2506
Shrink model to first 2507 iterations.

REPEAT 1/2 | FOLD 2/5
[500]	valid_0's rmse: 0.515216	valid_0's l2: 0.265447
0:	learn: 0.6048598	test: 0.6014568	best: 0.6014568 (0)	total: 81.5ms	remaining: 7m 3s
500:	learn: 0.5090549	test: 0.5165294	best: 0.5165294 (500)	total: 33.7s	remaining: 5m 15s
1000:	learn: 0.5027548	test: 0.5155970	best: 0.5155961 (999)	total: 1m 6s	remaining: 4m 39s
1500:	learn: 0.

## 9. Branch evaluation

In [16]:
N_REPEATS=len(CFG.REPEAT_SEEDS)
valid_branches=[]; oof_cont={}; test_cont={}
for b in branch_names:
    if np.all(OOF_COUNT[b]==N_REPEATS):
        valid_branches.append(b)
        oof_cont[b]=OOF_SUM[b]/N_REPEATS
        test_cont[b]=TEST_SUM[b]/(N_REPEATS*CFG.N_FOLDS)
    else:
        print('DROP incomplete branch',b,'OOF count',OOF_COUNT[b].min(),OOF_COUNT[b].max())

assert all(b in valid_branches for b in ['lgb_core','xgb_core','cat_core'])
print('Valid branches:',valid_branches)

branch_rows=[]
for b in valid_branches:
    p=oof_cont[b]
    # Cross-fitted thresholds only.
    cls=np.zeros(len(y),int)
    for f in range(CFG.N_FOLDS):
        cal=primary_fold_id!=f; val=primary_fold_id==f
        thr,_=optimize_thresholds(y[cal],p[cal],maxiter=140)
        cls[val]=apply_thresholds(p[val],thr)
    row={'branch':b,'crossfit_qwk':float(qwk(y,cls)),'rmse':rmse(y,p),**boundary_auc(y,p)}
    branch_rows.append(row)
branch_eval=pd.DataFrame(branch_rows).sort_values('crossfit_qwk',ascending=False)
display(branch_eval)

Valid branches: ['lgb_core', 'xgb_core', 'cat_core', 'xgb_categorical', 'lgb_cumulative_ordinal', 'catboost_allcat']


,branch,crossfit_qwk,rmse,auc_gt1,auc_gt2,auc_gt3,auc_1_vs_2
2,cat_core,0.479198,0.515737,0.779286,0.872577,0.809043,0.765000
1,xgb_core,0.478883,0.516200,0.778793,0.871905,0.807169,0.764522
4,lgb_cumulative_ordinal,0.478730,0.515934,0.779420,0.871915,0.806386,0.765218
5,catboost_allcat,0.478443,0.516219,0.778733,0.872114,0.808614,0.764427
0,lgb_core,0.477936,0.516038,0.778709,0.872184,0.807584,0.764422
3,xgb_categorical,0.477257,0.516726,0.777988,0.871729,0.808018,0.763658


## 10. Cross-fitted blend selection

The core three-model weights are calibrated only on the other four primary folds.  Sidecars are then added greedily only when they improve calibration QWK by at least `MIN_SIDECAR_GAIN`.  This is repeated separately for every held-out primary fold.

In [17]:
CORE=['lgb_core','xgb_core','cat_core']
SIDECARS=[b for b in ['lgb_cumulative_ordinal','xgb_categorical','catboost_allcat'] if b in valid_branches]

def core_weight_grid(step=0.10):
    vals=np.arange(step,1.0,step)
    out=[]
    for a in vals:
        for b in vals:
            c=1-a-b
            if c>=step-1e-9:
                out.append(np.array([a,b,c],float))
    out.append(np.array([1/3,1/3,1/3],float))
    # unique rounded rows
    seen=set(); unique=[]
    for w in out:
        k=tuple(np.round(w,6))
        if k not in seen: seen.add(k); unique.append(w)
    return unique

CORE_WEIGHT_GRID=core_weight_grid(CFG.CORE_WEIGHT_STEP)

def apply_recipe(indices,recipe):
    M=np.column_stack([oof_cont[b][indices] for b in CORE])
    p=M@np.asarray(recipe['core_weights'])
    for step in recipe['steps']:
        p=(1-step['alpha'])*p+step['alpha']*oof_cont[step['branch']][indices]
    return p

def fit_recipe(cal_idx,allowed_sidecars=None):
    allowed=SIDECARS if allowed_sidecars is None else [b for b in SIDECARS if b in allowed_sidecars]
    M=np.column_stack([oof_cont[b][cal_idx] for b in CORE])
    best=None
    for w in CORE_WEIGHT_GRID:
        p=M@w; thr,s=optimize_thresholds(y[cal_idx],p,maxiter=140)
        if best is None or s>best['score']:
            best={'core_weights':w.copy(),'score':float(s),'thresholds':thr.copy()}
    current=M@best['core_weights']; current_score=best['score']; steps=[]; remaining=list(allowed)

    for _ in range(CFG.MAX_SIDECARS):
        winner=None
        for b in remaining:
            for alpha in CFG.SIDECAR_ALPHA_GRID:
                cand=(1-alpha)*current+alpha*oof_cont[b][cal_idx]
                thr,s=optimize_thresholds(y[cal_idx],cand,maxiter=120)
                rec={'branch':b,'alpha':float(alpha),'score':float(s),'gain':float(s-current_score),'thresholds':thr.copy()}
                if winner is None or rec['score']>winner['score']: winner=rec
        if winner is None or winner['gain']<CFG.MIN_SIDECAR_GAIN: break
        current=(1-winner['alpha'])*current+winner['alpha']*oof_cont[winner['branch']][cal_idx]
        current_score=winner['score']; steps.append({'branch':winner['branch'],'alpha':winner['alpha'],'gain':winner['gain']})
        remaining.remove(winner['branch'])
        best['thresholds']=winner['thresholds']
    return {'core_weights':best['core_weights'].tolist(),'steps':steps,'thresholds':best['thresholds'].tolist(),'cal_qwk':float(current_score)}

# Core-only crossfit reference.
core_cross_class=np.zeros(len(y),int); core_cross_score=np.zeros(len(y),float)
core_fold_rows=[]
for f in range(CFG.N_FOLDS):
    cal=np.where(primary_fold_id!=f)[0]; val=np.where(primary_fold_id==f)[0]
    recipe=fit_recipe(cal,allowed_sidecars=[])
    p=apply_recipe(val,recipe)
    core_cross_score[val]=p; core_cross_class[val]=apply_thresholds(p,recipe['thresholds'])
    core_fold_rows.append({'fold':f,'val_qwk':qwk(y[val],core_cross_class[val]),'weights':recipe['core_weights']})
CORE_CROSSFIT_QWK=float(qwk(y,core_cross_class))
print('CORE CROSSFIT QWK:',CORE_CROSSFIT_QWK)

CORE CROSSFIT QWK: 0.47889859250843714


In [18]:
final_cross_class=np.zeros(len(y),int); final_cross_score=np.zeros(len(y),float)
blend_fold_rows=[]; selection_counter=Counter()
for f in range(CFG.N_FOLDS):
    cal=np.where(primary_fold_id!=f)[0]; val=np.where(primary_fold_id==f)[0]
    recipe=fit_recipe(cal)
    p=apply_recipe(val,recipe)
    final_cross_score[val]=p; final_cross_class[val]=apply_thresholds(p,recipe['thresholds'])
    for s in recipe['steps']: selection_counter[s['branch']]+=1
    blend_fold_rows.append({'fold':f,'cal_qwk':recipe['cal_qwk'],'val_qwk':float(qwk(y[val],final_cross_class[val])),
                            'core_weights':recipe['core_weights'],'steps':json.dumps(recipe['steps']),
                            'thresholds':recipe['thresholds']})

FINAL_CROSSFIT_QWK=float(qwk(y,final_cross_class))
print('FINAL CROSSFIT QWK:',FINAL_CROSSFIT_QWK)
print('GAIN VS CORE:',FINAL_CROSSFIT_QWK-CORE_CROSSFIT_QWK)
print('Sidecar fold-selection frequency:',dict(selection_counter))
print('Boundary AUC:',boundary_auc(y,final_cross_score))
display(pd.DataFrame(blend_fold_rows))

FINAL CROSSFIT QWK: 0.4788316526993418
GAIN VS CORE: -6.693980909533437e-05
Sidecar fold-selection frequency: {'lgb_cumulative_ordinal': 3, 'catboost_allcat': 1, 'xgb_categorical': 1}
Boundary AUC: {'auc_gt1': 0.7796106193835466, 'auc_gt2': 0.872803066845338, 'auc_gt3': 0.8083474818360749, 'auc_1_vs_2': 0.7653516502519236}


,fold,cal_qwk,val_qwk,core_weights,steps,thresholds
0,0,0.478185,0.495670,"[0.1, 0.30000000000000004, 0.6]","[{""branch"": ""lgb_cumulative_ordinal"", ""alpha"":...","[1.5529465409017855, 2.0036649387886705, 2.485..."
1,1,0.481964,0.479335,"[0.1, 0.5, 0.4]",[],"[1.55445003679823, 2.036047434237771, 2.497834..."
2,2,0.483158,0.475464,"[0.30000000000000004, 0.1, 0.6]","[{""branch"": ""catboost_allcat"", ""alpha"": 0.15, ...","[1.546863522618561, 2.0240712220637427, 2.5142..."
3,3,0.485330,0.464080,"[0.1, 0.8, 0.09999999999999998]","[{""branch"": ""lgb_cumulative_ordinal"", ""alpha"":...","[1.5563099034254384, 2.049797752225576, 2.5160..."
4,4,0.481451,0.478824,"[0.1, 0.2, 0.7]","[{""branch"": ""xgb_categorical"", ""alpha"": 0.45, ...","[1.541003539203071, 2.030159527688922, 2.54151..."


## 11. Unique-ID diagnostic and error analysis

In [19]:
oof_uid=(pd.DataFrame({'id':train[CFG.ID_COL].to_numpy(),'y':y,'score':final_cross_score})
         .groupby('id',as_index=False).agg(y=('y','first'),score=('score','mean')))
uid_thr,UID_DIAGNOSTIC_QWK=optimize_thresholds(oof_uid.y.to_numpy(),oof_uid.score.to_numpy(),maxiter=220)
print('Unique-ID diagnostic QWK:',UID_DIAGNOSTIC_QWK,'rows:',len(oof_uid))

cm=pd.DataFrame(confusion_matrix(y,final_cross_class,labels=[1,2,3,4]),index=[1,2,3,4],columns=[1,2,3,4])
display(cm)

# Adjacent 1<->2 error cohort, OOF only.
mask12=np.isin(y,[1,2])
cohort=pd.DataFrame({'y':y[mask12],'pred':final_cross_class[mask12],'score':final_cross_score[mask12]})
cohort['cohort']=np.where(cohort.y==cohort.pred,'correct_12',cohort.y.astype(str)+'_to_'+cohort.pred.astype(str))
display(cohort.cohort.value_counts().to_frame('rows'))

Unique-ID diagnostic QWK: 0.48016158773868045 rows: 94663


,1,2,3,4
1,30212,12023,677,56
2,14939,27603,5136,511
3,161,1308,1589,501
4,140,205,288,260


,rows
cohort,
correct_12,57815
2_to_1,14939
1_to_2,12023
2_to_3,5136
1_to_3,677
2_to_4,511
1_to_4,56


## 12. Stable global recipe and test prediction

For export, only sidecars selected in at least two of the five cross-fitted calibration recipes are eligible.  This prevents a single-fold sidecar win from controlling the test submission.

In [20]:
stable_sidecars=[b for b,n in selection_counter.items() if n>=CFG.MIN_GLOBAL_SIDECAR_FOLD_SELECTIONS]
print('Stable sidecars eligible for final export:',stable_sidecars)

global_recipe=fit_recipe(np.arange(len(train)),allowed_sidecars=stable_sidecars)
print('Global export recipe:',json.dumps(global_recipe,indent=2))

def apply_test_recipe(recipe):
    M=np.column_stack([test_cont[b] for b in CORE])
    p=M@np.asarray(recipe['core_weights'])
    for step in recipe['steps']:
        p=(1-step['alpha'])*p+step['alpha']*test_cont[step['branch']]
    return p

FINAL_TEST_CONT=apply_test_recipe(global_recipe)
FINAL_THRESHOLDS=np.asarray(global_recipe['thresholds'],float)

Stable sidecars eligible for final export: ['lgb_cumulative_ordinal']
Global export recipe: {
  "core_weights": [
    0.1,
    0.5,
    0.4
  ],
  "steps": [
    {
      "branch": "lgb_cumulative_ordinal",
      "alpha": 0.2,
      "gain": 0.00026412917168439076
    }
  ],
  "thresholds": [
    1.5516791809210828,
    2.0202286247216796,
    2.483905555595907
  ],
  "cal_qwk": 0.48176145150168903
}


## 13. Submission — unique supplied `test_final.id` contract

In [21]:
# Historical valid submissions for this competition use one row per unique supplied test_final.id.
# Duplicate IDs have identical feature payloads, so continuous scores are averaged before thresholding.
test_uid=(pd.DataFrame({'id':test[CFG.ID_COL].to_numpy(),'score':FINAL_TEST_CONT})
          .groupby('id',sort=False,as_index=False).score.mean())
test_uid[CFG.TARGET]=apply_thresholds(test_uid.score.to_numpy(),FINAL_THRESHOLDS).astype(int)
submission=test_uid[['id',CFG.TARGET]].copy()

assert len(submission)==CFG.EXPECTED_UNIQUE_TEST_IDS
assert submission['id'].is_unique
assert submission[CFG.TARGET].isin([1,2,3,4]).all()
assert submission.isna().sum().sum()==0
assert submission['id'].tolist()==test[CFG.ID_COL].drop_duplicates().tolist()

submission_path=CFG.OUTPUT_DIR/'submission_v6.csv'
submission.to_csv(submission_path,index=False)
print('Saved:',submission_path,'rows:',len(submission))
print(submission[CFG.TARGET].value_counts(normalize=True).sort_index())

Saved: /kaggle/working/oqdq5_v6_recovery_allcat/submission_v6.csv rows: 23666
life_satisfaction
1    0.469154
2    0.432857
3    0.082143
4    0.015846
Name: proportion, dtype: float64


## 14. Final gates and reproducibility artifacts

In [22]:
GAIN_VS_CORE=float(FINAL_CROSSFIT_QWK-CORE_CROSSFIT_QWK)
BEATS_048231_OFFLINE=bool(FINAL_CROSSFIT_QWK>=CFG.NEAREST_PUBLIC_REFERENCE)
TOP1_CANDIDATE=bool(FINAL_CROSSFIT_QWK>=CFG.TOP1_PUBLIC_REFERENCE)
WORTH_SUBMITTING=bool(BEATS_048231_OFFLINE and UID_DIAGNOSTIC_QWK>=0.4820 and GAIN_VS_CORE>=0)

print('CORE_CROSSFIT_QWK =',CORE_CROSSFIT_QWK)
print('FINAL_CROSSFIT_QWK =',FINAL_CROSSFIT_QWK)
print('UID_DIAGNOSTIC_QWK =',UID_DIAGNOSTIC_QWK)
print('GAIN_VS_CORE =',GAIN_VS_CORE)
print('BEATS_0.48231_OFFLINE =',BEATS_048231_OFFLINE)
print('WORTH_SUBMITTING =',WORTH_SUBMITTING)
print('TOP1_CANDIDATE_0.61541 =',TOP1_CANDIDATE)

CORE_CROSSFIT_QWK = 0.47889859250843714
FINAL_CROSSFIT_QWK = 0.4788316526993418
UID_DIAGNOSTIC_QWK = 0.48016158773868045
GAIN_VS_CORE = -6.693980909533437e-05
BEATS_0.48231_OFFLINE = False
WORTH_SUBMITTING = False
TOP1_CANDIDATE_0.61541 = False


In [23]:
feature_selection_df=pd.DataFrame(feature_selection_rows)
feature_importance_df=pd.DataFrame(feature_importance_rows)
if len(feature_importance_df):
    feature_importance_df=(feature_importance_df.groupby(['branch','feature'],as_index=False).importance.mean()
                           .sort_values(['branch','importance'],ascending=[True,False]))

fold_audit_df=pd.DataFrame(fold_audit)
blend_fold_df=pd.DataFrame(blend_fold_rows)
core_fold_df=pd.DataFrame(core_fold_rows)

oof_out=pd.DataFrame({'id':train[CFG.ID_COL].to_numpy(),CFG.TARGET:y,'primary_fold':primary_fold_id,
                      'oof_core_crossfit':core_cross_score,'oof_final_crossfit':final_cross_score,
                      'oof_final_class':final_cross_class})
for b in valid_branches: oof_out[f'oof_{b}']=oof_cont[b]

test_out=pd.DataFrame({'row_index':np.arange(len(test)),'id':test[CFG.ID_COL].to_numpy(),'pred_final_continuous':FINAL_TEST_CONT})
for b in valid_branches: test_out[f'test_{b}']=test_cont[b]

summary={
    'version':CFG.VERSION,
    'n_train':int(len(train)),'n_test_rows':int(len(test)),'n_test_unique_ids':int(test[CFG.ID_COL].nunique()),
    'selected_feature_budget':int(BUDGET),'selected_allcat_cutoff':int(ALLCAT_CUTOFF),
    'valid_branches':valid_branches,
    'branch_failures':{k:[list(x) for x in v] for k,v in branch_failures.items()},
    'core_crossfit_qwk':float(CORE_CROSSFIT_QWK),
    'final_crossfit_qwk':float(FINAL_CROSSFIT_QWK),
    'gain_vs_core':float(GAIN_VS_CORE),
    'unique_id_diagnostic_qwk':float(UID_DIAGNOSTIC_QWK),
    'boundary_auc':boundary_auc(y,final_cross_score),
    'stable_sidecars':stable_sidecars,
    'sidecar_fold_selection_frequency':dict(selection_counter),
    'global_export_recipe':global_recipe,
    'beats_0_48231_offline':bool(BEATS_048231_OFFLINE),
    'worth_submitting_gate':bool(WORTH_SUBMITTING),
    'top1_candidate_ge_0_61541':bool(TOP1_CANDIDATE),
    'submission_rows':int(len(submission)),
    'constraints':[
        'Only competition-provided files are used.',
        'No external data/source reconstruction/pretrained external foundation model/manual mapping/leaderboard probing.',
        '2x5 StratifiedGroupKFold with groups=id.',
        'No feature scaling for tree models.',
        'No SMOTE or random oversampling.',
        'Native missing values are preserved; only train-gap inferred continuous tail sentinels are separated.',
        'Feature budget and all-cat cutoff are selected on train-only nested probes.',
        'Final blend weights and thresholds are cross-fitted across primary grouped folds.',
        'Only sidecars selected in >=2 crossfit folds are eligible for test export.',
    ],
}

def json_default(obj):
    if isinstance(obj,np.generic): return obj.item()
    if isinstance(obj,np.ndarray): return obj.tolist()
    if isinstance(obj,Path): return str(obj)
    if isinstance(obj,(tuple,set)): return list(obj)
    return str(obj)

artifacts={
    'submission':submission_path,
    'summary':CFG.OUTPUT_DIR/'v6_summary.json',
    'oof':CFG.OUTPUT_DIR/'oof_v6.csv',
    'test_predictions':CFG.OUTPUT_DIR/'test_predictions_v6.csv',
    'design_probe':CFG.OUTPUT_DIR/'design_probe_v6.csv',
    'design_probe_summary':CFG.OUTPUT_DIR/'design_probe_summary_v6.csv',
    'branch_evaluation':CFG.OUTPUT_DIR/'branch_evaluation_v6.csv',
    'feature_selection':CFG.OUTPUT_DIR/'feature_selection_v6.csv',
    'feature_importance':CFG.OUTPUT_DIR/'feature_importance_v6.csv',
    'feature_provenance':CFG.OUTPUT_DIR/'feature_provenance_v6.csv',
    'sentinel_audit':CFG.OUTPUT_DIR/'sentinel_audit_v6.csv',
    'fold_audit':CFG.OUTPUT_DIR/'fold_audit_v6.csv',
    'blend_folds':CFG.OUTPUT_DIR/'blend_fold_recipes_v6.csv',
}

oof_out.to_csv(artifacts['oof'],index=False)
test_out.to_csv(artifacts['test_predictions'],index=False)
probe_df.to_csv(artifacts['design_probe'],index=False)
probe_summary.to_csv(artifacts['design_probe_summary'],index=False)
branch_eval.to_csv(artifacts['branch_evaluation'],index=False)
feature_selection_df.to_csv(artifacts['feature_selection'],index=False)
feature_importance_df.to_csv(artifacts['feature_importance'],index=False)
FEATURE_PROVENANCE.to_csv(artifacts['feature_provenance'],index=False)
sentinel_audit.to_csv(artifacts['sentinel_audit'],index=False)
fold_audit_df.to_csv(artifacts['fold_audit'],index=False)
blend_fold_df.to_csv(artifacts['blend_folds'],index=False)
with open(artifacts['summary'],'w') as f: json.dump(summary,f,indent=2,default=json_default)

print('Artifacts:')
for k,p in artifacts.items(): print(' -',k,p)
display(pd.Series(summary).to_frame('value'))

Artifacts:
 - submission /kaggle/working/oqdq5_v6_recovery_allcat/submission_v6.csv
 - summary /kaggle/working/oqdq5_v6_recovery_allcat/v6_summary.json
 - oof /kaggle/working/oqdq5_v6_recovery_allcat/oof_v6.csv
 - test_predictions /kaggle/working/oqdq5_v6_recovery_allcat/test_predictions_v6.csv
 - design_probe /kaggle/working/oqdq5_v6_recovery_allcat/design_probe_v6.csv
 - design_probe_summary /kaggle/working/oqdq5_v6_recovery_allcat/design_probe_summary_v6.csv
 - branch_evaluation /kaggle/working/oqdq5_v6_recovery_allcat/branch_evaluation_v6.csv
 - feature_selection /kaggle/working/oqdq5_v6_recovery_allcat/feature_selection_v6.csv
 - feature_importance /kaggle/working/oqdq5_v6_recovery_allcat/feature_importance_v6.csv
 - feature_provenance /kaggle/working/oqdq5_v6_recovery_allcat/feature_provenance_v6.csv
 - sentinel_audit /kaggle/working/oqdq5_v6_recovery_allcat/sentinel_audit_v6.csv
 - fold_audit /kaggle/working/oqdq5_v6_recovery_allcat/fold_audit_v6.csv
 - blend_folds /kaggle/worki

,value
version,v6_recovery_allcat
n_train,95609
n_test_rows,23902
n_test_unique_ids,23666
selected_feature_budget,140
selected_allcat_cutoff,50
valid_branches,"[lgb_core, xgb_core, cat_core, xgb_categorical..."
branch_failures,{}
core_crossfit_qwk,0.478899
final_crossfit_qwk,0.478832


## Decision guide

Use the notebook's own **cross-fitted** metrics, not the global OOF optimizer, to decide whether to submit.

- `FINAL_CROSSFIT_QWK < 0.4817`: V2 strength was not recovered; do not submit.
- `0.4817–0.4820`: recovered baseline but still weak.
- `0.4820–0.4823`: watch / one-slot only if other diagnostics are stable.
- `>= 0.48231` with non-negative core gain and stable sidecar selection: submission-worthy for the current near-term target.
- `>= 0.61541`: only then call it a genuine Top-1 offline candidate.

The public-reference numbers are **report-only** and are never used to choose features, hyperparameters, branches, blend weights, or thresholds.